In [ ]:
import os
import glob
import zipfile
import rasterio
import rasterio.mask
from rasterio.warp import transform_geom
import numpy as np
from rasterio.vrt import WarpedVRT

In [ ]:
BASE_DIR = "/content/drive/MyDrive/Galaxeye Task"
DATA_DIR = "/content/drive/MyDrive/Galaxeye Task/data"
PROCESSED_DIR = "/content/drive/MyDrive/Galaxeye Task/data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
# Morigaon / Bhuragaon Confluence
roi_coords = [[
    (92.35, 26.35), (92.55, 26.35),
    (92.55, 26.50), (92.35, 26.50),
    (92.35, 26.35)
]]
roi_geojson = {'type': 'Polygon', 'coordinates': roi_coords}

In [ ]:
def unzip_all():

    print("Starting Unzip Process...")
    for sensor in ["sentinel2", "sentinel1"]:
        sensor_dir = os.path.join(DATA_DIR, sensor)
        extract_path = os.path.join(sensor_dir, "extracted")

        # Make sure directory exists
        if not os.path.exists(sensor_dir):
            print(f"Warning: Directory not found: {sensor_dir}")
            continue

        zip_files = glob.glob(os.path.join(sensor_dir, "*.zip"))
        if not zip_files:
            print(f"No zip files found in {sensor}")
            continue

        print(f"  Found {len(zip_files)} zips in {sensor}. Unzipping...")
        for z in zip_files:
            with zipfile.ZipFile(z, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
    print("Unzipping complete.")

In [ ]:
def find_bands():

    print("Locating Bands...")
    # Find S2 Bands (Recursive)
    try:
        s2_g = glob.glob(os.path.join(DATA_DIR, "sentinel2", "extracted", "**", "*_B03_10m.jp2"), recursive=True)[0]
        s2_n = glob.glob(os.path.join(DATA_DIR, "sentinel2", "extracted", "**", "*_B08_10m.jp2"), recursive=True)[0]
        print("  Found Sentinel-2 Bands.")
    except IndexError:
        print("S2 bands not found. Check if zip file name matches pattern.")
        s2_g, s2_n = None, None

    # Find S1 Band
    try:
        s1_v = glob.glob(os.path.join(DATA_DIR, "sentinel1", "extracted", "**", "*vv*.tiff"), recursive=True)[0]
        print("  Found Sentinel-1 Band.")
    except IndexError:
        print("S1 band not found.")
        s1_v = None

    return s2_g, s2_n, s1_v

In [ ]:
def clip_and_save(src_path, out_path, roi_geom):
    """
    Clips the raster to the ROI.
    Auto-detects if the image is GCP-referenced (like Sentinel-1)
    and warps it on-the-fly to handle the clip.
    """
    if not src_path: return

    with rasterio.open(src_path) as src:

        # LOGIC CHECK: Does it have a CRS?
        if src.crs:
            # Case A: Standard Image (Sentinel-2) - Works as before
            print(f"  Processing {os.path.basename(src_path)} (Standard CRS)...")
            roi_transformed = transform_geom('EPSG:4326', src.crs, roi_geom)
            out_img, out_transform = rasterio.mask.mask(src, [roi_transformed], crop=True)
            out_meta = src.meta.copy()

        else:
            # Case B: Raw Sentinel-1 (GCPs) - NEEDS WARPED VRT
            print(f"  Processing {os.path.basename(src_path)} (GCP Mode - Auto-Warping)...")

            # We create a virtual "Corrected" version of the image in WGS84 (Lat/Lon)
            with WarpedVRT(src, crs='EPSG:4326') as vrt:
                # Now the VRT has a CRS, so we can transform the ROI to match IT
                roi_transformed = transform_geom('EPSG:4326', vrt.crs, roi_geom)

                # Clip the VRT instead of the raw src
                out_img, out_transform = rasterio.mask.mask(vrt, [roi_transformed], crop=True)
                out_meta = vrt.meta.copy()

        # Common Save Logic
        out_meta.update({
            "driver": "GTiff",
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform,
            "dtype": rasterio.float32,
            "count": 1 # Force single band
        })

        # Save
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(out_img[0].astype(rasterio.float32), 1)

    print(f"  Saved: {os.path.basename(out_path)}")

In [ ]:
# --- 3. EXECUTION ---
unzip_all() # No arguments needed now
s2_g, s2_n, s1_v = find_bands()

if s2_g and s1_v:
    print("\nClipping to ROI...")
    clip_and_save(s2_g, os.path.join(PROCESSED_DIR, "s2_green_clipped.tif"), roi_geojson)
    clip_and_save(s2_n, os.path.join(PROCESSED_DIR, "s2_nir_clipped.tif"), roi_geojson)
    clip_and_save(s1_v, os.path.join(PROCESSED_DIR, "s1_sar_clipped.tif"), roi_geojson)
    print("\nAll data processed and ready for analysis.")
else:
    print("\nCannot proceed. Files missing.")

Starting Unzip Process...
  Found 1 zips in sentinel2. Unzipping...
  Found 1 zips in sentinel1. Unzipping...
Unzipping complete.
Locating Bands...
  Found Sentinel-2 Bands.
  Found Sentinel-1 Band.

Clipping to ROI...
  Processing T46RDQ_20240301T042709_B03_10m.jp2 (Standard CRS)...
  Saved: s2_green_clipped.tif
  Processing T46RDQ_20240301T042709_B08_10m.jp2 (Standard CRS)...
  Saved: s2_nir_clipped.tif
  Processing s1a-iw-grd-vv-20240723t115715-20240723t115740-054888-06af59-001.tiff (GCP Mode - Auto-Warping)...
  Saved: s1_sar_clipped.tif

All data processed and ready for analysis.
